# **Image Caption Generator (Using CNN+ LSTM)**

In [ ]:
import numpy as np
import pandas as pd
import os
import string
import pickle

from tqdm import tqdm

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

In [ ]:
!wget -q https://github.com/Avaneesh40585/Flickr8k-Dataset/releases/download/v1.0/Flickr8k_Dataset.zip  # Images
!wget -q https://github.com/Avaneesh40585/Flickr8k-Dataset/releases/download/v1.0/Flickr8k_text.zip     # Text annotations

In [ ]:
!unzip -qq Flickr8k_Dataset.zip
!unzip -qq Flickr8k_text.zip

In [ ]:
image_dir = "/content/Flicker8k_Dataset"

In [ ]:
import os
len(os.listdir(image_dir))

8091

In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.models import Model

In [ ]:
base_model = InceptionV3(weights='imagenet')
model = Model(inputs=base_model.input, outputs=base_model.layers[-2].output)

96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
features = {}

for img_name in tqdm(os.listdir(image_dir)):
    filename = image_dir + "/" + img_name

    image = load_img(filename, target_size=(299, 299))  # 🔴 change
    image = img_to_array(image)
    image = np.expand_dims(image, axis=0)
    image = preprocess_input(image)

    feature = model.predict(image, verbose=0)
    image_id = img_name.split('.')[0]
    features[image_id] = feature


100%|██████████| 8091/8091 [50:29<00:00,  2.67it/s]


In [ ]:
pickle.dump(features, open("features_inception.pkl", "wb"))

In [ ]:
def load_doc(filename):
    with open(filename, 'r') as f:
        text = f.read()
    return text

doc = load_doc("/content/Flickr8k.token.txt")

In [ ]:
mapping = {}

for line in doc.split('\n'):
    tokens = line.split()
    if len(tokens) < 2:
        continue

    image_id = tokens[0].split('.')[0]
    caption = ' '.join(tokens[1:])

    if image_id not in mapping:
        mapping[image_id] = []
    mapping[image_id].append(caption)


In [ ]:
len(mapping)

8092

In [ ]:
def clean(mapping):
    for key, captions in mapping.items():
        for i in range(len(captions)):
            caption = captions[i]
            caption = caption.lower()
            caption = caption.translate(str.maketrans('', '', string.punctuation))
            captions[i] = "startseq " + caption + " endseq"

clean(mapping)

In [ ]:
all_captions = []
for key in mapping:
    for caption in mapping[key]:
        all_captions.append(caption)

len(all_captions)

40460

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)

vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

8831


In [ ]:
max_length = max(len(c.split()) for c in all_captions)
print(max_length)

38


In [ ]:
def data_generator(mapping, features, tokenizer, max_length, vocab_size):
    while True:
        for key, captions in mapping.items():
            feature = features[key][0]

            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]

                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]

                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]

                    yield (np.array([feature]), np.array([in_seq])), np.array([out_seq])


In [ ]:
# image feature input
inputs1 = Input(shape=(2048,))
fe1 = Dropout(0.4)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

# text input
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.4)(se1)
se3 = LSTM(256)(se2)

# decoder
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 38)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 38, 256)   │  2,260,736 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 2048)      │          0 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 38, 256)   │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 38)        │          0 │ input_layer_4[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │    524,544 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 256)       │    525,312 │ dropout_3[0][0],  │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256)       │          0 │ dense_3[0][0],    │
│                     │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │     65,792 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 8831)      │  2,269,567 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,645,951 (21.54 MB)

 Trainable params: 5,645,951 (21.54 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs = 10
steps = 500
generator = data_generator(mapping, features, tokenizer, max_length, vocab_size)
model.fit(generator, epochs=epochs, steps_per_epoch=steps, verbose=1)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 50s 99ms/step - loss: 3.6422
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 50s 100ms/step - loss: 3.5791
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 50s 100ms/step - loss: 4.4124
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 4.4531
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 50s 101ms/step - loss: 4.6577
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 50s 100ms/step - loss: 5.3540
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 49s 99ms/step - loss: 5.2735
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 50s 100ms/step - loss: 5.0887
Epoch 9/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 51s 102ms/step - loss: 5.2196
Epoch 10/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 50s 101ms/step - loss: 5.6443


In [ ]:
model.save("caption_model.h5")

In [ ]:
def idx_to_word(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None

In [ ]:
def generate_caption(model, feature, tokenizer, max_length):
    in_text = "startseq"
    for i in range(max_length):
        seq = tokenizer.texts_to_sequences([in_text])[0]
        seq = pad_sequences([seq], maxlen=max_length)

        yhat = model.predict([feature, seq], verbose=0)
        yhat = np.argmax(yhat)

        word = idx_to_word(yhat, tokenizer)
        if word is None:
            break

        in_text += " " + word
        if word == "endseq":
            break

    result = in_text.split()
    final = result[1:-1]
    return ' '.join(final)

In [ ]:
import random

key = random.choice(list(mapping.keys()))
feature = features[key]
print(generate_caption(model, feature, tokenizer, max_length))

a boy in a climbing
